# E-Commerce Customer Analytics - EDA

This notebook performs exploratory data analysis on the Online Retail II dataset.

Goals:
- Validate dataset quality
- Inspect customer and revenue patterns
- Build intuition for RFM clustering, CLV regression, and churn classification

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path.cwd().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_preprocessing import load_raw_data, clean_transactions, preprocessing_summary

In [ ]:
raw_path = PROJECT_ROOT.parent / 'online_retail_II.csv'
processed_path = PROJECT_ROOT / 'data' / 'processed_online_retail_II.csv'

raw_df = load_raw_data(raw_path)
clean_df = clean_transactions(raw_df)
summary_df = preprocessing_summary(raw_df, clean_df)

processed_path.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(processed_path, index=False)

summary_df

In [ ]:
clean_df.head()

In [ ]:
clean_df.describe(include='all').transpose().head(20)

In [ ]:
missing = clean_df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
monthly_revenue = (
    clean_df.assign(InvoiceMonth=clean_df['InvoiceDate'].dt.to_period('M').dt.to_timestamp())
    .groupby('InvoiceMonth')['TotalAmount']
    .sum()
    .sort_index()
)

ax = monthly_revenue.plot(title='Monthly Revenue Trend')
ax.set_ylabel('Revenue')
plt.show()

In [ ]:
top_countries = (
    clean_df.groupby('Country', as_index=False)['TotalAmount']
    .sum()
    .sort_values('TotalAmount', ascending=False)
    .head(10)
)

sns.barplot(data=top_countries, x='TotalAmount', y='Country', palette='viridis')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue')
plt.ylabel('Country')
plt.show()

In [ ]:
customer_stats = clean_df.groupby('CustomerID').agg(
    invoices=('Invoice', 'nunique'),
    avg_order_value=('TotalAmount', 'mean'),
    total_spend=('TotalAmount', 'sum'),
)

customer_stats.describe().transpose()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(customer_stats['total_spend'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Distribution: Customer Total Spend')
axes[0].set_xlabel('Total Spend')

sns.histplot(customer_stats['invoices'], bins=40, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Distribution: Number of Invoices per Customer')
axes[1].set_xlabel('Invoices')

plt.tight_layout()
plt.show()

## EDA Takeaways

- Revenue concentration by a subset of countries and customers suggests strong segmentation potential.
- Customer spend and purchase frequency are right-skewed, supporting robust scaling/modeling choices.
- Time trend reveals seasonality that can influence CLV and churn labeling windows.